# Fichas técnicas en PDF → mismo esquema que `DATOS_TEC` + embeddings

Objetivo del recomendador: recibir la ficha (PDF) de una grasa o aceite —p. ej. de la competencia o la que usa
el cliente— y recomendar los productos Interlub que mejor encajan. Este notebook prepara las dos piezas:

1. **Extracción** (IA): cada PDF de `data/raw/pdfs/` se procesa con el prompt de
   `scripts/prompts/extraccion_pdfs.md` → `data/processed/competencia_extraccion.json` (valores literales).
2. **Normalización** (`scripts/fichas_pdf.py`): unidades (N → kgf, °C(°F) → °C), coma decimal, `>`/`<`/`~`/rangos
   (columnas `_tipo`, como en `DATOS_TEC_clean`), vocabulario de espesante/aceite base/color/textura.
   → `data/processed/competencia_clean.csv`
3. **Embeddings** (`paraphrase-multilingual-MiniLM-L12-v2`, por lotes) → `data/processed/embeddings.npz`:
   - `ficha`: el mismo texto plantilla para productos Interlub y de PDF (comparables entre sí).
   - `texto`: texto libre de los PDF (descripción, aplicaciones, beneficios, certificaciones), para consultas
     en lenguaje natural. `DATOS_TEC` no tiene texto libre equivalente.

Para agregar un PDF nuevo: extraerlo con el prompt, añadir el objeto al JSON y volver a correr este notebook.

In [ ]:
# fichas_pdf.py y recomendador.py viven en scripts/
import sys

sys.path.insert(0, "../scripts")

In [ ]:
import json

import fichas_pdf as F
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

EXTRACCION = "../data/processed/competencia_extraccion.json"
INTERLUB = "../data/processed/DATOS_TEC_clean.csv"
OUT = "../data/processed/competencia_clean.csv"
EMB = "../data/processed/embeddings.npz"
MODELO = "paraphrase-multilingual-MiniLM-L12-v2"

with open(EXTRACCION, encoding="utf-8") as f:
    registros = json.load(f)
interlub = pd.read_csv(INTERLUB)
print(len(registros), "productos extraídos de", len({r["archivo_fuente"] for r in registros}), "PDFs")

In [ ]:
# 1. Normalize to the DATOS_TEC schema (one row per product/grade)
comp = F.normalizar(registros)
print(comp.shape, "| vigentes:", comp["vigente"].sum())
print("columnas fuera de DATOS_TEC:", sorted(set(comp.columns) - set(interlub.columns)))
cols = ["nombre_producto", "vigente", "familia", "espesante", "aceite_base", "grado_nlgi_consistencia",
        "grado_nlgi_por_penetracion", "temperatura_de_servicio_c_min", "temperatura_de_servicio_c_max",
        "punto_de_gota_c", "viscosidad_del_aceite_base_a_40c_cst", "punto_de_soldadura_cuatro_bolas_kgf",
        "color", "tiene_registro_nsf"]
comp[cols]

In [ ]:
# 2. Validation: values outside the DATOS_TEC vocabulary/ranges and the same consistency rules as 01_limpieza.ipynb.
#    These are warnings (the PDF may be right); the recommender should just skip unknown attributes.
avisos = {
    "espesante fuera del vocabulario de Interlub": comp["espesante_en_vocabulario"].eq(False),
    "aceite base no identificado": comp["aceite_base"].isna(),
    "corrosion al cobre sin clase 1a/1b": comp["corrosion_al_cobre"].isna(),
    "nlgi declarado != nlgi por penetracion": (comp["grado_nlgi_consistencia"] - comp["grado_nlgi_por_penetracion"]).abs().ge(1)
        & ~(comp["grado_nlgi_consistencia"].eq(0) & comp["grado_nlgi_por_penetracion"].lt(0)),
    "punto de gota < temperatura maxima de servicio": comp["punto_de_gota_c"] < comp["temperatura_de_servicio_c_max"],
    "sin temperatura de servicio": comp["temperatura_de_servicio_c_max"].isna(),
}
for col in ["punto_de_gota_c", "viscosidad_del_aceite_base_a_40c_cst", "punto_de_soldadura_cuatro_bolas_kgf",
            "penetracion_de_cono_a_25c_0_1mm", "temperatura_de_servicio_c_max"]:
    lo, hi = interlub[col].min(), interlub[col].max()
    avisos[f"{col} fuera del rango de Interlub [{lo:g}, {hi:g}]"] = (comp[col] < lo) | (comp[col] > hi)

pd.DataFrame({k: ", ".join(comp.loc[v.fillna(False), "nombre_producto"]) for k, v in avisos.items()}, index=["productos"]).T

In [ ]:
# 3. Export (data/processed/*.csv is git-ignored; encrypt with scripts/dvc_encrypt.sh to share)
comp.to_csv(OUT, index=False)
print("saved:", comp.shape, "| read back:", pd.read_csv(OUT).shape)

In [ ]:
# 4. Texts to embed. "ficha" uses one template for both sources; "texto" is the PDFs' own free text.
interlub_ficha = F.texto_ficha(interlub)
comp_ficha = F.texto_ficha(comp)
comp_texto = F.texto_libre(comp)
print("Interlub:", interlub_ficha.iloc[0], "\n")
print("PDF:", comp_ficha.iloc[-1], "\n")
print("texto libre (palabras por producto):", comp_texto.str.split().str.len().describe()[["min", "mean", "max"]].to_dict())

In [ ]:
# 5. Embeddings in batches, L2-normalized (cosine similarity = dot product).
#    Free text is embedded per sentence: the model reads only ~128 tokens and averaging long fields diluted the
#    key phrases (a food-grade query did not find the only H1 grease). Product-level text vector = mean of its
#    sentences; a query scores each product by its best sentence (step 7).
modelo = SentenceTransformer(MODELO)
print("max tokens:", modelo.max_seq_length)


def embed(textos):
    return modelo.encode(list(textos), batch_size=32, normalize_embeddings=True, show_progress_bar=False)


frag = F.fragmentos(comp)
frag_emb = embed(frag["texto"])
texto_emb = np.stack([frag_emb[frag["producto"].to_numpy() == i].mean(axis=0) for i in comp.index])
texto_emb /= np.linalg.norm(texto_emb, axis=1, keepdims=True)

emb = {
    "interlub_id": interlub["codigoproducto"].to_numpy(),
    "interlub_ficha": embed(interlub_ficha),
    "competencia_id": comp["nombre_producto"].to_numpy(),
    "competencia_ficha": embed(comp_ficha),
    "competencia_texto": texto_emb,
    "fragmento_producto": frag["producto"].to_numpy(),
    "fragmento_texto": frag["texto"].to_numpy(),
    "fragmento_emb": frag_emb,
}
np.savez(EMB, modelo=MODELO, **emb)
print(len(frag), "fragmentos de texto")
{k: v.shape for k, v in emb.items()}

In [ ]:
# 6. Sanity check (not the recommender yet): closest Interlub products to each current PDF product using only
#    the "ficha" embedding. Shown next to the key specs to judge whether text similarity tracks the specs.
sim = emb["competencia_ficha"] @ emb["interlub_ficha"].T
key = ["familia", "espesante", "aceite_base", "grado_nlgi_por_penetracion", "temperatura_de_servicio_c_max",
       "viscosidad_del_aceite_base_a_40c_cst"]
filas = []
for i in np.flatnonzero(comp["vigente"]):
    filas.append(comp.loc[[i], ["nombre_producto", *key]].assign(similitud=np.nan))
    top = np.argsort(-sim[i])[:3]
    filas.append(interlub.iloc[top][["codigoproducto", *key]].rename(columns={"codigoproducto": "nombre_producto"})
                 .assign(similitud=sim[i, top].round(3)))
pd.concat(filas, ignore_index=True)

In [ ]:
# 7. Free-text queries against the PDFs' text (Spanish queries vs English/Spanish sheets): best sentence per product.
#    Write queries with accents: the model was trained on natural text and scores worse without them.
def buscar(consulta, k=3):
    s = pd.Series(emb["fragmento_emb"] @ embed([consulta])[0], index=frag.index)
    mejor = s.groupby(frag["producto"]).idxmax()
    out = pd.DataFrame({"nombre_producto": comp.loc[mejor.index, "nombre_producto"].to_numpy(),
                        "similitud": s[mejor].round(3).to_numpy(), "frase": frag.loc[mejor, "texto"].to_numpy()})
    return out.sort_values("similitud", ascending=False).head(k).assign(consulta=consulta)


consultas = [
    "grasa para la industria alimentaria con contacto incidental con alimentos",
    "grasa para rodamientos de camiones con frenos de disco a alta temperatura",
    "grasa para engranajes abiertos en minería con mucha agua",
    "grasa tixotrópica para mecanismos con oscilaciones pequeñas",
]
pd.concat([buscar(c) for c in consultas], ignore_index=True).set_index(["consulta", "nombre_producto"])